# บทที่ 11: Searching (การค้นหา) และสรุปรวบยอด

> **รายวิชา:** Data Structures and Algorithms (สำหรับนิสิต IT ชั้นปีที่ 2)
> **อ้างอิงแนวเนื้อหา:** Narasimha Karumanchi, *Data Structures and Algorithmic Thinking with Python*
> **วิธีใช้:** อ่านคำอธิบาย → รันโค้ดทีละเซลล์ → ลองแก้ค่า input → ทำแบบฝึกหัดท้ายบท (เฉลยอยู่ท้ายไฟล์)

---

## 11.1 Linear vs Binary Search

| | Linear search | Binary search |
|---|---|---|
| ข้อมูลต้องเรียงก่อน | ไม่ | **ใช่** |
| Complexity | \(O(n)\) | \(O(\log n)\) |
| เหมาะกับ | ข้อมูลน้อย/ไม่เรียง, linked list | อาร์เรย์เรียงแล้ว, ค้นบ่อย |

**ข้อควรระวังของ binary search**
- คำนวณจุดกลางด้วย `mid = lo + (hi - lo) // 2` (กัน overflow ในภาษาอื่น)
- ระวัง infinite loop จากการอัปเดตขอบเขตผิด
- ถ้าต้องเรียงก่อน (\(O(n\log n)\)) แล้วค้นเพียงครั้งเดียว → linear search อาจคุ้มกว่า

In [ ]:
def linear_search(a, target):
    for i, x in enumerate(a):
        if x == target: return i
    return -1

def binary_search(a, target):
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        if a[mid] == target: return mid
        if a[mid] < target:  lo = mid + 1
        else:                hi = mid - 1
    return -1

def binary_search_rec(a, target, lo=0, hi=None):
    if hi is None: hi = len(a) - 1
    if lo > hi: return -1
    mid = lo + (hi - lo) // 2
    if a[mid] == target: return mid
    if a[mid] < target:  return binary_search_rec(a, target, mid + 1, hi)
    return binary_search_rec(a, target, lo, mid - 1)

a = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("linear(23) =", linear_search(a, 23))
print("binary(23) =", binary_search(a, 23), "| binary(7) =", binary_search(a, 7))
print("recursive  =", binary_search_rec(a, 91))

In [ ]:
# Binary search แบบขอบเขต (lower/upper bound) - ใช้บ่อยกว่าการหาค่าตรง ๆ
def lower_bound(a, x):
    # ตำแหน่งแรกที่ a[i] >= x  (เท่ากับ bisect.bisect_left)
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x: lo = mid + 1
        else:          hi = mid
    return lo

def upper_bound(a, x):
    # ตำแหน่งแรกที่ a[i] > x  (เท่ากับ bisect.bisect_right)
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] <= x: lo = mid + 1
        else:           hi = mid
    return lo

def count_occurrences(a, x):
    return upper_bound(a, x) - lower_bound(a, x)      # O(log n)

import bisect
b = [1, 2, 2, 2, 3, 5, 5, 9]
print("lower_bound(2) =", lower_bound(b, 2), "| bisect_left =", bisect.bisect_left(b, 2))
print("upper_bound(2) =", upper_bound(b, 2), "| bisect_right =", bisect.bisect_right(b, 2))
print("จำนวน 2 =", count_occurrences(b, 2), "| จำนวน 5 =", count_occurrences(b, 5))

In [ ]:
# Binary search บน "คำตอบ" (binary search on answer) - รูปแบบที่ออกสอบบ่อย
def sqrt_int(n):
    # หา floor(sqrt(n)) โดยไม่ใช้ math.sqrt -> O(log n)
    lo, hi, ans = 0, n, 0
    while lo <= hi:
        mid = (lo + hi) // 2
        if mid * mid <= n:
            ans, lo = mid, mid + 1
        else:
            hi = mid - 1
    return ans

def search_rotated(a, target):
    # อาร์เรย์เรียงแล้วที่ถูกหมุน เช่น [4,5,6,7,0,1,2] -> ยังค้นได้ O(log n)
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if a[mid] == target: return mid
        if a[lo] <= a[mid]:                       # ครึ่งซ้ายเรียงปกติ
            if a[lo] <= target < a[mid]: hi = mid - 1
            else:                        lo = mid + 1
        else:                                     # ครึ่งขวาเรียงปกติ
            if a[mid] < target <= a[hi]: lo = mid + 1
            else:                        hi = mid - 1
    return -1

def min_capacity_to_ship(weights, days):
    # โจทย์คลาสสิก: หาความจุน้อยสุดที่ส่งของให้เสร็จภายใน days วัน
    def feasible(cap):
        d, cur = 1, 0
        for w in weights:
            if cur + w > cap: d, cur = d + 1, w
            else:             cur += w
        return d <= days
    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid): hi = mid
        else:             lo = mid + 1
    return lo

print("sqrt_int(50) =", sqrt_int(50))
print("search_rotated =", search_rotated([4,5,6,7,0,1,2], 0))
print("min capacity =", min_capacity_to_ship([1,2,3,4,5,6,7,8,9,10], 5))

## 11.2 สรุปรวบยอด: เลือกโครงสร้างข้อมูลอย่างไร

| ความต้องการ | โครงสร้างที่เหมาะ | เหตุผล |
|---|---|---|
| เข้าถึงตำแหน่งที่ i เร็ว | Array / list | index access \(O(1)\) |
| แทรก/ลบที่หัวบ่อย | Linked list, `deque` | \(O(1)\) ไม่ต้องเลื่อนข้อมูล |
| ค้นด้วย key เร็วมาก | Hash table (`dict`, `set`) | เฉลี่ย \(O(1)\) |
| ต้องการข้อมูลเรียงตลอดเวลา | BST (balanced) | inorder ได้ลำดับ, ค้น \(O(\log n)\) |
| หยิบค่ามาก/น้อยสุดซ้ำ ๆ | Heap (`heapq`) | peek \(O(1)\), pop \(O(\log n)\) |
| ประมวลผลย้อนกลับ / undo | Stack | LIFO |
| จัดคิวงานตามลำดับมาก่อน | Queue | FIFO |
| ความสัมพันธ์แบบเครือข่าย | Graph | BFS/DFS/Dijkstra |
| จัดกลุ่ม/ตรวจสมาชิกกลุ่ม | Union-Find | union & find \(\approx O(1)\) |

**คำแนะนำในการฝึก:** อ่านโจทย์ → ระบุ operation ที่ต้องทำ *บ่อยที่สุด* → เลือกโครงสร้างที่ทำ operation นั้นได้ถูกที่สุด → ประเมิน complexity ก่อนเขียนโค้ด

---
## แบบฝึกหัดบทที่ 11

1. หาค่าที่ปรากฏเกินครึ่งของอาร์เรย์ (majority element) ด้วย \(O(n)\) time / \(O(1)\) space
2. หา peak element (ตัวที่มากกว่าเพื่อนบ้านทั้งสองข้าง) ด้วย \(O(\log n)\)
3. หาตัวเลขที่หายไป 1 ตัวจาก 0..n
4. ค้นค่าในเมทริกซ์ที่แต่ละแถวและคอลัมน์เรียงแล้ว ด้วย \(O(m+n)\)

### เฉลยบทที่ 11

In [ ]:
# ข้อ 1: Boyer-Moore voting
def majority_element(a):
    cand, cnt = None, 0
    for x in a:
        if cnt == 0: cand, cnt = x, 1
        elif x == cand: cnt += 1
        else: cnt -= 1
    return cand if a.count(cand) > len(a)//2 else None

# ข้อ 2: binary search โดยเดินไปทางฝั่งที่สูงขึ้น
def find_peak(a):
    lo, hi = 0, len(a) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < a[mid + 1]: lo = mid + 1
        else:                   hi = mid
    return lo

# ข้อ 3: ผลรวมที่ควรได้ - ผลรวมจริง (หรือใช้ XOR ก็ได้)
def missing_number(a):
    n = len(a)
    return n * (n + 1) // 2 - sum(a)

# ข้อ 4: เริ่มมุมขวาบน แล้วตัดแถว/คอลัมน์ทีละครั้ง -> O(m+n)
def search_matrix(M, target):
    r, c = 0, len(M[0]) - 1
    while r < len(M) and c >= 0:
        if M[r][c] == target: return (r, c)
        if M[r][c] > target:  c -= 1
        else:                 r += 1
    return None

print(majority_element([2,2,1,2,3,2,2]))
print("peak index:", find_peak([1,3,20,4,1,0]))
print("missing:", missing_number([3,0,1]))
print("matrix:", search_matrix([[1,4,7],[2,5,8],[3,6,9]], 5))

---
## แหล่งอ้างอิงและอ่านเพิ่มเติม

- Narasimha Karumanchi, *Data Structures and Algorithmic Thinking with Python: Data Structure and Algorithmic Puzzles* — [ฉบับ PDF ที่ใช้อ้างอิงในรายวิชา](https://github.com/Paradorn-248/Computer-Programming/blob/master/Data%20Structure%20and%20Algorithmic%20Thinking%20with%20Python%20%20Data%20Structure%20and%20Algorithmic%20Puzzles%20(%20PDFDrive%20).pdf)
- [Python DSA tutorial ของ W3Schools](https://www.w3schools.com/python/python_dsa.asp) — ทบทวนแนวคิดพื้นฐานพร้อมภาพประกอบ
- [เอกสาร heapq ของ Python](https://docs.python.org/3/library/heapq.html) และ [collections.deque](https://docs.python.org/3/library/collections.html#collections.deque)
- [Time complexity ของ operation ต่าง ๆ ใน Python](https://wiki.python.org/moin/TimeComplexity) — ตารางอ้างอิงที่ควรให้นิสิตดูคู่กับบทที่ 2